# Fraud Detection Complete System
Pipeline + Interactive Dashboard (with ngrok) + Automation


In [1]:
# Installing dependencies
import subprocess, sys, pkg_resources, importlib

required = {'dash', 'plotly', 'pandas', 'numpy', 'scikit-learn', 'openpyxl'}
installed = {pkg.key for pkg in pkg_resources.working_set}
missing = required - installed
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *missing])

try:
    import ngrok
except ImportError:
    try:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'pyngrok'])
    except:
        pass

print('All dependencies ready')

/tmp/ipykernel_6832/2136762823.py:2: DeprecationWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html
  import subprocess, sys, pkg_resources, importlib


All dependencies ready


In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import IsolationForest
from datetime import datetime
import os
import json
import warnings
import threading
warnings.filterwarnings('ignore')

DATA_FILE = '/content/bank-transactions-data-2.csv'
RESULTS_FILE = 'fraud_detection_results.csv'
# PIPELINE
class FraudDetectionPipeline:
    def __init__(self, contamination=0.1):
        self.contamination = contamination
        self.scaler = StandardScaler()
        self.label_encoders = {}
        self.iso_forest = None
        self.df_processed = None
        self.feature_columns = None

    def load_data(self, filepath):
        df = pd.read_csv(filepath)
        print(f'Dataset Shape: {df.shape}')
        return df

    def preprocess_data(self, df):
        print('\nTASK 1: DATA PREPROCESSING')
        df_processed = df.copy()
        df_processed.fillna(method='ffill', inplace=True)
        df_processed.fillna(0, inplace=True)
        date_columns = ['TransactionDate', 'PreviousTransactionDate']
        for col in date_columns:
            if col in df_processed.columns:
                df_processed[col] = pd.to_datetime(df_processed[col], dayfirst=True, errors='coerce')
        if 'TransactionDate' in df_processed.columns:
            df_processed['TransactionHour'] = df_processed['TransactionDate'].dt.hour
            df_processed['TransactionDay'] = df_processed['TransactionDate'].dt.day
            df_processed['TransactionMonth'] = df_processed['TransactionDate'].dt.month
            df_processed['TransactionDayOfWeek'] = df_processed['TransactionDate'].dt.dayofweek
        if 'PreviousTransactionDate' in df_processed.columns:
            df_processed['TimeSincePreviousTxn'] = (df_processed['TransactionDate'] - df_processed['PreviousTransactionDate']).dt.total_seconds() / 3600
        if 'TransactionAmount' in df_processed.columns:
            df_processed['LogAmount'] = np.log1p(df_processed['TransactionAmount'])
            if 'CustomerID' in df_processed.columns:
                customer_avg = df_processed.groupby('CustomerID')['TransactionAmount'].transform('mean')
                df_processed['AmountDeviationFromAvg'] = df_processed['TransactionAmount'] - customer_avg
        df_processed['is_safe'] = 0
        if all(col in df_processed.columns for col in ['TransactionAmount', 'TransactionHour']):
            safe_rule1 = ((df_processed['TransactionAmount'] < 50) & (df_processed['TransactionHour'].between(9, 17)))
            df_processed.loc[safe_rule1, 'is_safe'] = 1
        if 'Channel' in df_processed.columns:
            safe_rule2 = df_processed['Channel'] == 'Mobile Banking'
            df_processed.loc[safe_rule2, 'is_safe'] = 1
        print(f'Safe transactions marked: {df_processed["is_safe"].sum()}')
        self.df_processed = df_processed
        return df_processed

    def prepare_features(self, df_processed):
        categorical_cols = df_processed.select_dtypes(include=['object']).columns.tolist()
        numerical_cols = df_processed.select_dtypes(include=[np.number]).columns.tolist()
        exclude_cols = ['TransactionID', 'CustomerID', 'is_safe', 'is_anomaly', 'combined_anomaly', 'iso_anomaly', 'z_anomaly', 'anomaly_score']
        numerical_cols = [col for col in numerical_cols if col not in exclude_cols]
        categorical_cols = [col for col in categorical_cols if col not in exclude_cols]
        df_encoded = df_processed.copy()
        for col in categorical_cols:
            if col in df_encoded.columns:
                if col not in self.label_encoders:
                    le = LabelEncoder()
                    df_encoded[col] = le.fit_transform(df_encoded[col].astype(str))
                    self.label_encoders[col] = le
                else:
                    le = self.label_encoders[col]
                    df_encoded[col] = le.transform(df_encoded[col].astype(str))
        self.feature_columns = numerical_cols + categorical_cols
        return df_encoded[self.feature_columns]

    def detect_anomalies(self, df_processed):
        print('\nTASK 2: ANOMALY DETECTION')
        X = self.prepare_features(df_processed)
        X_scaled = self.scaler.fit_transform(X)
        print('Training Isolation Forest...')
        self.iso_forest = IsolationForest(contamination=self.contamination, random_state=42, n_estimators=100, max_features=1.0, bootstrap=False)
        iso_predictions = self.iso_forest.fit_predict(X_scaled)
        df_processed = df_processed.copy()
        df_processed['iso_anomaly'] = (iso_predictions == -1).astype(int)
        print('Applying Z-score method...')
        z_scores = np.abs((X_scaled - X_scaled.mean(axis=0)) / (X_scaled.std(axis=0) + 1e-10))
        df_processed['z_anomaly'] = (z_scores > 3).any(axis=1).astype(int)
        df_processed['combined_anomaly'] = ((df_processed['iso_anomaly'] == 1) | (df_processed['z_anomaly'] == 1)).astype(int)
        df_processed.loc[df_processed['is_safe'] == 1, 'combined_anomaly'] = 0
        df_processed.loc[df_processed['is_safe'] == 1, 'iso_anomaly'] = 0
        df_processed.loc[df_processed['is_safe'] == 1, 'z_anomaly'] = 0
        scores = self.iso_forest.score_samples(X_scaled)
        df_processed['anomaly_score'] = scores
        print(f'Total: {len(df_processed)}, Anomalies: {df_processed["combined_anomaly"].sum()} ({(df_processed["combined_anomaly"].sum()/len(df_processed)*100):.2f}%)')
        self.df_processed = df_processed
        return df_processed

    def save_results(self, output_path=RESULTS_FILE):
        if self.df_processed is not None:
            self.df_processed.to_csv(output_path, index=False)
            print(f'Results saved to {output_path}')

    def run_pipeline(self, filepath, save_output=True):
        print('FRAUD DETECTION PIPELINE - STARTING')
        df = self.load_data(filepath)
        df_processed = self.preprocess_data(df)
        df_processed = self.detect_anomalies(df_processed)
        if save_output:
            self.save_results()
        print('PIPELINE COMPLETED')
        return df_processed


# AUTOMATION
class FraudMonitoringAutomation:
    def __init__(self, config_file='automation_config.json'):
        self.config = self.load_config(config_file)
        self.pipeline = FraudDetectionPipeline(contamination=self.config.get('contamination', 0.1))
        self.log_file = self.config.get('log_file', 'fraud_monitoring.log')

    def load_config(self, config_file):
        default_config = {
            'data_source': DATA_FILE,
            'output_dir': 'daily_reports',
            'contamination': 0.1,
            'alert_threshold': 5,
            'email_enabled': False,
            'log_file': 'fraud_monitoring.log'
        }
        if os.path.exists(config_file):
            with open(config_file, 'r') as f:
                default_config.update(json.load(f))
        else:
            with open(config_file, 'w') as f:
                json.dump(default_config, f, indent=2)
        return default_config

    def log(self, message, level='INFO'):
        timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        log_message = f'[{timestamp}] [{level}] {message}'
        print(log_message)
        with open(self.log_file, 'a') as f:
            f.write(log_message + '\n')

    def process_new_data(self):
        self.log('Starting fraud detection pipeline...')
        df_results = self.pipeline.run_pipeline(self.config['data_source'], save_output=True)
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        output_dir = self.config['output_dir']
        os.makedirs(output_dir, exist_ok=True)
        df_results.to_csv(os.path.join(output_dir, f'fraud_results_{timestamp}.csv'), index=False)
        self.log(f'Results saved')
        return df_results, timestamp

    def generate_excel_report(self, df_results, timestamp):
        try:
            from openpyxl import Workbook
            from openpyxl.styles import Font, PatternFill, Alignment
            output_dir = self.config['output_dir']
            report_file = os.path.join(output_dir, f'fraud_report_{timestamp}.xlsx')
            wb = Workbook()
            ws = wb.active
            ws.title = 'Summary'
            header_fill = PatternFill(start_color='3498DB', end_color='3498DB', fill_type='solid')
            header_font = Font(color='FFFFFF', bold=True, size=14)
            total_txns = len(df_results)
            total_anomalies = df_results['combined_anomaly'].sum()
            anomaly_rate = (total_anomalies / total_txns * 100) if total_txns > 0 else 0
            summary_data = [
                ['Metric', 'Value'],
                ['Report Generated', datetime.now().strftime('%Y-%m-%d %H:%M:%S')],
                ['Total Transactions', total_txns],
                ['Flagged Anomalies', total_anomalies],
                ['Anomaly Rate', f'{anomaly_rate:.2f}%'],
                ['Total Volume', f'${df_results["TransactionAmount"].sum():,.2f}'],
            ]
            for row_idx, row_data in enumerate(summary_data, 1):
                for col_idx, value in enumerate(row_data, 1):
                    cell = ws.cell(row=row_idx, column=col_idx, value=value)
                    if row_idx == 1:
                        cell.fill = header_fill
                        cell.font = header_font
            wb.save(report_file)
            self.log(f'Excel report: {report_file}')
            return report_file
        except Exception as e:
            self.log(f'Excel error: {e}', level='ERROR')
            return None

    def run_daily_monitoring(self):
        self.log('STARTING DAILY MONITORING')
        try:
            df_results, timestamp = self.process_new_data()
            self.generate_excel_report(df_results, timestamp)
            self.log('DAILY MONITORING COMPLETED')
        except Exception as e:
            self.log(f'Failed: {e}', level='ERROR')

In [3]:
# DASHBOARD with LLM Explanation on click
from dash import Dash, dcc, html, Input, Output
import plotly.express as px

class FraudDashboard:
    def __init__(self, data_path=RESULTS_FILE):
        self.data_path = data_path
        self.df = None
        self.app = None
        self.load_data()

    def load_data(self):
        print(f'Loading data from {self.data_path}...')
        self.df = pd.read_csv(self.data_path)
        date_cols = ['TransactionDate', 'PreviousTransactionDate']
        for col in date_cols:
            if col in self.df.columns:
                self.df[col] = pd.to_datetime(self.df[col], errors='coerce')
        print(f'Loaded {len(self.df)} transactions, {self.df["combined_anomaly"].sum()} anomalies')

    def generate_explanation(self, row):
        reasons = []
        amount = row.get('TransactionAmount', 0)
        hour = row.get('TransactionHour', 12)
        score = row.get('anomaly_score', 0)
        channel = row.get('Channel', 'Unknown')
        is_anomaly = row.get('combined_anomaly', 0)
        if is_anomaly:
            if amount > 1000:
                reasons.append(f'High transaction amount (${amount:.2f}) exceeds typical range')
            if hour < 6 or hour > 22:
                reasons.append(f'Unusual transaction time at {hour}:00 (outside normal hours)')
            if score < -0.5:
                reasons.append('Transaction pattern deviates significantly from normal behavior')
            time_since = row.get('TimeSincePreviousTxn', 0)
            if time_since < 0.5 and time_since > 0:
                reasons.append(f'Rapid transaction -- only {time_since:.1f} hours since last transaction')
            if not reasons:
                reasons.append('Anomaly detected by Isolation Forest and Z-score ensemble')
            return ' | '.join(reasons)
        else:
            return 'Transaction appears normal with no suspicious indicators.'

    def create_dashboard(self):
        self.app = Dash(__name__)
        channels = sorted(self.df['Channel'].unique()) if 'Channel' in self.df.columns else []

        self.app.layout = html.Div([
            html.Div([
                html.H1('Fraud Detection Dashboard', style={'textAlign': 'center', 'color': '#2c3e50', 'marginBottom': '30px'}),
                html.P('AI-Powered Transaction Monitoring System', style={'textAlign': 'center', 'color': '#7f8c8d', 'fontSize': '18px'})
            ], style={'padding': '10px', 'backgroundColor': '#ecf0f1'}),

            html.Div([
                html.Div([html.H3(f'{len(self.df):,}', style={'color': '#3498db', 'margin': '0'}), html.P('Total Transactions', style={'margin': '5px 0'})], style={'flex': '1', 'padding': '20px', 'backgroundColor': 'white', 'margin': '10px', 'borderRadius': '10px', 'boxShadow': '0 2px 4px rgba(0,0,0,0.1)'}),
                html.Div([html.H3(f"{self.df['combined_anomaly'].sum():,}", style={'color': '#e74c3c', 'margin': '0'}), html.P('Flagged Anomalies', style={'margin': '5px 0'})], style={'flex': '1', 'padding': '20px', 'backgroundColor': 'white', 'margin': '10px', 'borderRadius': '10px', 'boxShadow': '0 2px 4px rgba(0,0,0,0.1)'}),
                html.Div([html.H3(f"{(self.df['combined_anomaly'].sum()/len(self.df)*100):.2f}%", style={'color': '#f39c12', 'margin': '0'}), html.P('Anomaly Rate', style={'margin': '5px 0'})], style={'flex': '1', 'padding': '20px', 'backgroundColor': 'white', 'margin': '10px', 'borderRadius': '10px', 'boxShadow': '0 2px 4px rgba(0,0,0,0.1)'}),
                html.Div([html.H3(f"${self.df['TransactionAmount'].sum():,.2f}", style={'color': '#27ae60', 'margin': '0'}), html.P('Total Volume', style={'margin': '5px 0'})], style={'flex': '1', 'padding': '20px', 'backgroundColor': 'white', 'margin': '10px', 'borderRadius': '10px', 'boxShadow': '0 2px 4px rgba(0,0,0,0.1)'}),
            ], style={'display': 'flex', 'justifyContent': 'space-around', 'margin': '10px'}),

            html.Div([
                html.H3('Filters', style={'marginBottom': '10px'}),
                html.Div([html.Label('Channel:', style={'fontWeight': 'bold', 'marginRight': '10px'}), dcc.Dropdown(id='channel-filter', options=[{'label': 'All Channels', 'value': 'ALL'}]+[{'label': ch, 'value': ch} for ch in channels], value='ALL', style={'width': '250px'})], style={'display': 'inline-block', 'marginRight': '30px'}),
                html.Div([html.Label('Show:', style={'fontWeight': 'bold', 'marginRight': '10px'}), dcc.Dropdown(id='anomaly-filter', options=[{'label': 'All Transactions', 'value': 'all'}, {'label': 'Anomalies Only', 'value': 'anomaly'}, {'label': 'Normal Only', 'value': 'normal'}], value='all', style={'width': '200px'})], style={'display': 'inline-block'}),
            ], style={'padding': '20px', 'backgroundColor': 'white', 'margin': '20px', 'borderRadius': '10px', 'boxShadow': '0 2px 4px rgba(0,0,0,0.1)'}),

            html.Div([
                html.Div([
                    html.H2('Anomaly Analysis', style={'color': '#2c3e50', 'marginBottom': '10px'}),
                    dcc.Graph(id='anomaly-scatter'),
                ], style={'flex': '1', 'padding': '20px', 'backgroundColor': 'white', 'margin': '10px', 'borderRadius': '10px', 'boxShadow': '0 2px 4px rgba(0,0,0,0.1)'}),

                html.Div([
                    html.H2('Transaction Explanation', style={'color': '#2c3e50', 'marginBottom': '10px'}),
                    html.Div(id='txn-explanation', style={
                        'padding': '20px',
                        'backgroundColor': '#f8f9fa',
                        'borderRadius': '10px',
                        'minHeight': '100px',
                        'fontSize': '15px',
                        'lineHeight': '1.6',
                        'color': '#7f8c8d',
                        'fontStyle': 'italic'
                    })
                ], style={'flex': '1', 'padding': '20px', 'backgroundColor': 'white', 'margin': '10px',
                         'borderRadius': '10px', 'boxShadow': '0 2px 4px rgba(0,0,0,0.1)'}),
            ], style={'display': 'flex', 'flexDirection': 'row'}),
        ], style={'fontFamily': 'Arial, sans-serif', 'backgroundColor': '#f5f6fa', 'minHeight': '100vh'})

        @self.app.callback(Output('anomaly-scatter', 'figure'), [Input('channel-filter', 'value'), Input('anomaly-filter', 'value')])
        def update_dashboard(channel, anomaly_filter):
            filtered_df = self.df.copy()
            if channel != 'ALL':
                filtered_df = filtered_df[filtered_df['Channel'] == channel]
            if anomaly_filter == 'anomaly':
                filtered_df = filtered_df[filtered_df['combined_anomaly'] == 1]
            elif anomaly_filter == 'normal':
                filtered_df = filtered_df[filtered_df['combined_anomaly'] == 0]
            fig = px.scatter(filtered_df, x='TransactionAmount', y='anomaly_score',
                           color='combined_anomaly',
                           title='Click any point for explanation',
                           labels={'combined_anomaly': 'Anomaly'},
                           color_discrete_map={0: '#3498db', 1: '#e74c3c'},
                           hover_data=['TransactionID', 'Channel'],
                           custom_data=['TransactionID'])
            fig.update_traces(marker=dict(size=10))
            return fig

        @self.app.callback(Output('txn-explanation', 'children'), Input('anomaly-scatter', 'clickData'))
        def show_explanation(clickData):
            if clickData is None:
                return 'Click on a point in the scatter plot to see an explanation for that transaction.'
            txn_id = clickData['points'][0]['customdata'][0]
            transaction = self.df[self.df['TransactionID'] == txn_id].iloc[0]
            explanation = self.generate_explanation(transaction)
            return html.Div([
                html.H4(f"Transaction {transaction['TransactionID']}", style={'color': '#2c3e50', 'marginBottom': '10px'}),
                html.Div([
                    html.Span(f"${transaction['TransactionAmount']:.2f}", style={'color': '#e74c3c', 'fontWeight': 'bold', 'fontSize': '18px'}),
                    html.Span(f"  |  {transaction['Channel']}  |  ", style={'color': '#7f8c8d'}),
                    html.Span(f"Score: {transaction['anomaly_score']:.3f}", style={'color': '#f39c12', 'fontWeight': 'bold'}),
                ], style={'marginBottom': '12px'}),
                html.Hr(),
                html.P(explanation, style={'backgroundColor': 'white', 'padding': '15px', 'borderRadius': '5px', 'borderLeft': '4px solid #e74c3c', 'color': '#2c3e50', 'fontStyle': 'normal'})
            ])

        print('Dashboard created successfully')

In [6]:
# MAIN: Run Pipeline -> Dashboard
import threading

# Step 1: Run the pipeline
print('STEP 1: RUNNING FRAUD DETECTION PIPELINE')
pipeline = FraudDetectionPipeline(contamination=0.1)
df_results = pipeline.run_pipeline(DATA_FILE)

print('\nSample Anomalies:')
anomalies = df_results[df_results['combined_anomaly'] == 1].head(10)
print(anomalies[['TransactionID', 'TransactionAmount', 'Channel', 'anomaly_score']].to_string(index=False))

# Step 2: Run automation
print('\n')
print('STEP 2: AUTOMATION WORKFLOW')
automation = FraudMonitoringAutomation()
automation.run_daily_monitoring()

# Step 3: Create dashboard
print('\n')
print('STEP 3: STARTING DASHBOARD')
dashboard = FraudDashboard(data_path=RESULTS_FILE)
dashboard.create_dashboard()

# Step 4: Start ngrok tunnel (for Colab/external access)
ngrok_url = None
try:
    from pyngrok import ngrok
    # Kill any existing tunnels
    ngrok.kill()
    ngrok.set_auth_token('3DgLFwHtUALTD3a4F3T21dHqUms_3K86Pygpdy7GAAcYiF6Qu')
    # Open tunnel on port 8050
    tunnel = ngrok.connect(8050, 'http')
    ngrok_url = tunnel.public_url
    print(f'\nNgrok public URL: {ngrok_url}')
    print('Share this URL to access the dashboard from anywhere')
except ImportError:
    print('\npyngrok not installed. Dashboard only available locally.')
    print('Install with: pip install pyngrok')
except Exception as e:
    print(f'\nNgrok error: {e}. Dashboard available locally.')

print('Dashboard is running!')
if ngrok_url:
    print(f'Public URL: {ngrok_url}')

dashboard.app.run(host='127.0.0.1', port=8050, debug=False)

STEP 1: RUNNING FRAUD DETECTION PIPELINE
FRAUD DETECTION PIPELINE - STARTING
Loading data from /content/bank-transactions-data-2.csv...
Dataset Shape: (2512, 16)

TASK 1: DATA PREPROCESSING
Safe transactions marked: 330

TASK 2: ANOMALY DETECTION
Training Isolation Forest...
Applying Z-score method...
Total: 2512, Anomalies: 259 (10.31%)
Results saved to fraud_detection_results.csv
PIPELINE COMPLETED

Sample Anomalies:
TransactionID  TransactionAmount Channel  anomaly_score
     TX000024             345.84  Online      -0.559424
     TX000027             246.93     ATM      -0.545180
     TX000033             396.45     ATM      -0.532919
     TX000035             255.26  Branch      -0.560125
     TX000075            1212.51  Branch      -0.507171
     TX000086            1340.19  Online      -0.579383
     TX000089               9.61  Online      -0.575893
     TX000092             223.85     ATM      -0.527235
     TX000102              98.54  Online      -0.547982
     TX000114    

INFO:dash.dash:Dash is running on http://127.0.0.1:8050/



 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:8050
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [23/May/2026 18:44:18] "GET / HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [23/May/2026 18:44:19] "GET /_dash-component-suites/dash/deps/polyfill@7.v4_1_0m1779561724.12.1.min.js HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [23/May/2026 18:44:19] "GET /_dash-component-suites/dash/deps/react@18.v4_1_0m1779561724.3.1.min.js HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [23/May/2026 18:44:19] "GET /_dash-component-suites/dash/dcc/dash_core_components.v4_1_0m1779561724.js HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [23/May/2026 18:44:19] "GET /_dash-component-suites/dash/dcc/dash_core_components-shared.v4_1_0m1779561724.js HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [23/May/2026 18:44:19] "GET /_dash-component-suites/dash/html/dash_html_components.v4_1_0m1779561724.m